# EDA final · cancelación de reservas de hotel

**Máster en IA, Cloud Computing y DevOps** · Machine Learning y Deep Learning · Práctica final

Si `eda_inicial.ipynb` es la cocina, este es el escaparate: aquí **no se explora**, se
*presenta*. Solo entra lo que sobrevivió a la exploración y **cambió una decisión** del
proyecto. Cada sección termina en una frase que empieza por «Decisión:».

Es el notebook que se abre en la defensa, así que tiene que ejecutarse de arriba abajo
sin errores y **con las salidas guardadas**: un notebook sin outputs obliga al profesor a
ejecutarlo para ver tu trabajo.

> La tabla del final es literalmente la del apartado 3 del README. Se rellena una vez, aquí.

In [ ]:
import sys, pathlib

# Subir hasta la raiz del proyecto (la carpeta que contiene src/), la abras desde donde la abras.
raiz = next(d for d in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (d / "src").is_dir())
if str(raiz) not in sys.path:
    sys.path.insert(0, str(raiz))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src import config

pd.set_option("display.max_columns", 40)
sns.set_theme(style="whitegrid")

In [ ]:
df = pd.read_csv(config.DATA_RAW, na_values=["NULL", "null", "", " "])
print(f"{df.shape[0]:,} filas x {df.shape[1]} columnas".replace(",", "."))
df.head()

## 1. El objetivo y su desbalanceo

El número que hunde a `accuracy` como métrica principal: un modelo que dijera siempre
«no cancela» ya acierta el 62,96 % sin haber aprendido nada.

In [ ]:
reparto = df[config.OBJETIVO].value_counts(normalize=True).sort_index()
print(reparto.mul(100).round(2).to_string())
print(f"razon {reparto[0] / reparto[1]:.2f} : 1")

# TODO: grafico de barras del reparto, con el % encima de cada barra.

**Decisión:** _(métrica principal = F1 de la clase positiva; accuracy queda como secundaria)_

## 2. La fuga: la columna que es la respuesta con otro nombre

`reservation_status` determina `is_canceled` al 100 %: `Check-Out` → 0, `Canceled` y
`No-Show` → 1. Es la etiqueta escrita de otra manera.

In [ ]:
tabla = pd.crosstab(df["reservation_status"], df[config.OBJETIVO])
display(tabla)

# La prueba: ninguna fila de la tabla tiene las dos columnas a la vez distintas de cero.
print("determina el objetivo al 100 %?", bool(((tabla > 0).sum(axis=1) == 1).all()))

**Decisión:** _(se eliminan `reservation_status` y `reservation_status_date`; quedan 29 predictoras)_

## 3. Nulos, duplicados e imposibles

Cuánto se pierde en cada paso de la limpieza. Estos números van al README: «se
eliminaron N filas» sin decir cuántas ni por qué no vale.

In [ ]:
# TODO: contar y tabular
#   - nulos por columna (con na_values puesto), en %
#   - duplicados exactos: df.duplicated().sum()
#   - imposibles: adr < 0  y  adults + children + babies == 0
# Deja el numero de filas que cae en CADA paso: va al apartado 3 del README.

**Decisión:** _(…)_

## 4. Cardinalidad: qué pasa si haces one-hot de todo

`country`, `agent` y `company` tienen cientos de valores distintos. One-hot a lo bruto
convierte 29 columnas en más de mil, y el árbol empieza a partir por «es de Portugal».

In [ ]:
categoricas = df.select_dtypes(include="object").columns
display(df[categoricas].nunique().sort_values(ascending=False).to_frame("valores distintos"))

# TODO: cuantas columnas saldrian de un get_dummies() ingenuo.

**Decisión:** _(agrupar en top-N + «otros» / frecuencia / target encoding dentro del CV — elige y justifica)_

## 5. Dónde está la señal: tasa de cancelación por grupo

Lo que de verdad se le enseña al profesor. Una variable es útil si la tasa de
cancelación **cambia mucho** entre sus categorías; si todas rondan el 37 %, no aporta.

In [ ]:
def tasa_por(col, minimo=500):
    """Tasa de cancelacion por categoria, ignorando las categorias con pocos casos."""
    g = df.groupby(col)[config.OBJETIVO].agg(["mean", "size"])
    return g[g["size"] >= minimo].sort_values("mean", ascending=False)

# TODO: hazlo para deposit_type, market_segment, customer_type, hotel, country (top 15).
# deposit_type = "Non Refund" suele salir por encima del 99 %: mirala y explicala.

**Decisión:** _(…)_

## 6. Las numéricas que más separan

`lead_time` es la candidata clara: cuanto antes reservas, más margen hay para arrepentirse.

In [ ]:
# TODO: distribucion de lead_time separada por is_canceled (dos histogramas o un KDE),
# y la tabla de medianas por clase. Recorta la cola larga para que se lea.

**Decisión:** _(…)_

## 7. La tabla que va al README

El apartado 3 del README se puntúa por lo que **cambió**, no por el número de gráficos.
Un hallazgo sin decisión al lado no suma.

In [ ]:
hallazgos = pd.DataFrame([
    # ("Hallazgo", "Evidencia (numero concreto de este notebook)", "Decision que tomamos"),
], columns=["Hallazgo", "Evidencia", "Decision"])

print(hallazgos.to_markdown(index=False))  # copiar y pegar en el README